# Italian LLM Evaluation - Model Evaluation Template

Use this notebook when you want to evaluate a real model checkpoint after the quick smoke path is already working.


This notebook keeps the same design principle as the repository:

- the notebook is only a launcher
- all evaluation logic stays in the package
- results are written to normal run directories


In [ ]:
REPO_URL = "https://github.com/<your-org-or-user>/it_eval_autoregressive_llms.git"
REPO_DIR = "it_eval_autoregressive_llms"

# Example: public HF repo id or a Drive-mounted local checkpoint path.
MODEL_SOURCE = "your-org/your-italian-base-model"
TOKENIZER_SOURCE = ""
MODEL_REVISION = "main"

# Leave as None to auto-select cuda when Colab exposes a GPU, otherwise cpu.
MODEL_DEVICE = None
MODEL_DTYPE = "auto"
MODEL_BATCH_SIZE = 1

HF_TOKEN = ""


In [ ]:
import os
import shutil
from pathlib import Path

%cd /content

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

repo_path = Path("/content") / REPO_DIR
if repo_path.exists():
    shutil.rmtree(repo_path)

!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/{REPO_DIR}

In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel
!python -m pip install "lighteval[multilingual]==0.13.0" --no-deps
!python -m pip install -r constraints/lighteval-python310-313.txt
!python -m pip install -e .[dev] --no-deps

In [ ]:
import torch

detected_device = "cuda" if torch.cuda.is_available() else "cpu"
selected_device = MODEL_DEVICE or detected_device

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"cuda device count: {torch.cuda.device_count()}")
    print(f"cuda device name: {torch.cuda.get_device_name(0)}")
    print(f"cuda capability: {torch.cuda.get_device_capability(0)}")
else:
    print("No CUDA GPU is available in this runtime. In Colab, switch Runtime > Change runtime type > GPU.")


In [ ]:
from pathlib import Path
import yaml

config_path = Path("configs/colab_model_eval.yaml")
config_payload = {
    "run_name": "colab_model_eval",
    "model": {
        "source": MODEL_SOURCE,
        "revision": MODEL_REVISION,
        "tokenizer_source": TOKENIZER_SOURCE or MODEL_SOURCE,
        "dtype": MODEL_DTYPE,
        "device": selected_device,
        "batch_size": MODEL_BATCH_SIZE,
    },
    "output": {
        "root_dir": "evaluation_results",
        "overwrite": False,
        "save_details": True,
    },
    "runtime": {
        "seed": 13,
        "python_executable": "python",
        "lighteval_command": "lighteval",
    },
    "lighteval": {
        "enabled": True,
        "suite": "quick",
        "dataset_loading_processes": 1,
    },
    "blimp_it": {
        "enabled": True
    },
    "generation": {
        "enabled": True,
        "prompts_path": "configs/generation_prompts.yaml",
        "seed": 13,
    },
}

with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config_payload, handle, allow_unicode=True, sort_keys=False)

print(config_path)
print(config_path.read_text(encoding="utf-8"))

In [ ]:
!python -m it_eval_framework.runners.run_lighteval --config configs/colab_model_eval.yaml

In [ ]:
!python -m it_eval_framework.runners.run_blimp_it --config configs/colab_model_eval.yaml

In [ ]:
!python -m it_eval_framework.runners.run_generation --config configs/colab_model_eval.yaml

In [ ]:
from pathlib import Path

run_dirs = sorted(Path("evaluation_results").rglob("run_config.yaml"))
latest_run_dir = run_dirs[-1].parent
print(latest_run_dir)
!python -m it_eval_framework.reporting.aggregate_results --run-dir "$latest_run_dir"

## Recommended usage

- keep this notebook for interactive model checks
- once the config looks right, move the same config back into the repo as a normal YAML file
- use the CLI runners for reproducible repeated runs
